# Suite Setup — Ensure demo-catalog exists

Creates `demo-catalog` and the `sentinel2-l2a` collection if they are not already present.
This notebook must run first (it sorts before `assets/` because `_` < `a` in ASCII).

In [ ]:
import os
import json
import time

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN = (
    os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
CATALOG_ID = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "sentinel2-l2a")

client = httpx.Client(
    base_url=BASE_URL,
    headers={"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"},
    timeout=120.0,
)

# 1. Ensure catalog exists
r = client.get(f"/stac/catalogs/{CATALOG_ID}")
if r.status_code == 200:
    print(f"Catalog '{CATALOG_ID}' already exists.")
else:
    for attempt in range(3):
        r = client.post("/stac/catalogs", content=json.dumps({
            "id": CATALOG_ID,
            "type": "Catalog",
            "title": "Demo Catalog",
            "description": "Demo catalog for notebook integration tests.",
            "stac_version": "1.0.0",
        }))
        if r.status_code == 201:
            print(f"Catalog '{CATALOG_ID}' created.")
            break
        if attempt < 2:
            print(f"Catalog create attempt {attempt+1} got {r.status_code}, retrying...")
            time.sleep(5 * (attempt + 1))
    else:
        raise RuntimeError(f"Failed to create catalog after 3 attempts: {r.status_code}: {r.text}")

    # Apply catalog-level defaults
    r2 = client.put(f"/configs/catalogs/{CATALOG_ID}/bulk", content=json.dumps({"configs": {
        "CollectionPostgresqlDriverConfig": {"enabled": True, "collection_type": "VECTOR"},
        "CollectionRoutingConfig": {"enabled": True, "operations": {
            "WRITE": [{"driver_id": "CollectionPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
            "READ":  [{"driver_id": "CollectionPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
        }},
    }}), timeout=60.0)
    print(f"Catalog defaults applied: {r2.status_code}")

# 2. Ensure collection exists
r = client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
if r.status_code == 200:
    print(f"Collection '{COLLECTION_ID}' already exists.")
else:
    r = client.post(f"/stac/catalogs/{CATALOG_ID}/collections", content=json.dumps({
        "id": COLLECTION_ID,
        "type": "Collection",
        "stac_version": "1.0.0",
        "title": "Sentinel-2 L2A",
        "description": "Test collection for notebook integration tests.",
        "license": "proprietary",
        "extent": {
            "spatial": {"bbox": [[-180, -90, 180, 90]]},
            "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
        },
        "links": [],
    }))
    assert r.status_code == 201, f"Collection create failed: {r.status_code}: {r.text}"
    print(f"Collection '{COLLECTION_ID}' created.")

print(f"\nSetup complete: {CATALOG_ID}/{COLLECTION_ID} ready.")
client.close()